In [144]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.append(str(root / "src"))

In [145]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    "roneneldan/TinyStoriesInstruct",
    "TinyStories-Instruct-valid.txt",
    repo_type="dataset",
)
path

'/home/rahul/.cache/huggingface/hub/datasets--roneneldan--TinyStoriesInstruct/snapshots/ee050ed1f8720795be342921335e821856a2b42e/TinyStories-Instruct-valid.txt'

In [146]:
txt = Path(path).read_text(encoding="utf-8")
recs = [r.strip() for r in txt.split("<|endoftext|>") if r.strip()]
print(len(recs))
print(recs[1])

25027
Random sentence: They are very excited and want to fly too.
Features: Dialogue
Summary: Tom and Anna are excited to go on a holiday with their parents, and they fly on a big plane to a place with sun and sand.
Story: 
Tom and Anna are brother and sister. They like to play with their toys and read books. They are very happy because they are going on a holiday with their mum and dad. They will fly on a big plane to a place with a lot of sun and sand.
The day of the holiday comes and they pack their bags. They go to the airport and wait for their plane. They see many other planes flying in the sky. They are very excited and want to fly too.
"Look, Anna, that plane is so big and fast!" Tom says.
"Yes, Tom, and it has wings and a tail. I wonder where it is going," Anna says.
They hear their mum call them. "Come on, kids, it's time to board our plane. We have to show our tickets and go through the gate."
They follow their mum and dad and get on their plane. They find their seats and bu

In [147]:
print(recs[2])

Features: Dialogue, BadEnding
Summary: Ben and Mia get lost in a subway station after a train zooms past them, leaving them alone and scared.
Random sentence: They wished they had never been eager to ride the subway.
Story: 
Ben and Mia were eager to ride the subway with their mom. They had never been on a subway before. They wanted to see the big trains and the tunnels and the people.
Mom bought the tickets and they went to the platform. They saw a subway coming. It was loud and fast. Ben and Mia held mom's hands. They waited for the subway to stop.
But the subway did not stop. It zoomed past them. Ben and Mia were scared. They let go of mom's hands and ran away. They did not see where they were going. They got lost in the crowd.
Mom shouted, "Ben! Mia! Stop! Come back!" But they did not hear her. They did not see her. They were alone and afraid.
Mom looked for them. She asked the people. She asked the workers. She asked the police. But no one had seen them. She cried and cried. She d

In [148]:
after = [
    r
    for r in recs
    if "Summary:" in r and "Story:" in r and r.index("Summary:") > r.index("Story:")
]
print(len(after), len(recs))
print(f'{len(after) / len(recs) * 100:.2f}%')

7560 25027
30.21%


In [149]:
print(after[0])

Words: meet, waffle, new
Story: 
Lily was hungry. She wanted a waffle. She asked her mom for a waffle. Her mom said, "OK, but you have to wait. I have to make it first."
Lily waited and waited. She saw a new toy on the table. It was a red car. She liked cars. She wanted to play with it. She took the car and ran to the living room.
She saw a big dog on the couch. The dog was sleeping. Lily wanted to meet the dog. She said, "Hi, dog. I have a new car. Do you want to see it?" She went closer to the dog.
The dog woke up. He did not like Lily. He did not like her car. He was angry. He growled and barked. He bit Lily's hand. Lily screamed and dropped the car.
Her mom heard her scream. She ran to the living room. She saw the dog and Lily. She was scared. She grabbed Lily and took her to the bathroom. She wrapped her hand with a cloth. She called the doctor.
The doctor came and looked at Lily's hand. He said, "She needs stitches. The dog was very bad. He hurt her a lot." He took Lily to the ho

In [150]:
FIELDS = ("Random sentence:", "Features:", "Words:", "Summary:", "Story:")

def parse(rec):
    fields, cur = {}, None
    for line in rec.split("\n"):
        hit = next((f for f in FIELDS if line.startswith(f)), None)
        if hit:
            cur = hit[:-1]
            fields[cur] = [line[len(hit):].strip()]
        elif cur:
            fields[cur].append(line.strip())
    return {k: "\n".join(v).strip() for k, v in fields.items()}

parsed = [parse(r) for r in recs]

In [151]:
parsed[0]

{'Summary': "Sam's tower of books falls down while his mom reads him a story about a dragon and a knight, but his mom comforts him and they continue to enjoy the story together."}

In [152]:
parsed[1]

{'Random sentence': 'They are very excited and want to fly too.',
 'Features': 'Dialogue',
 'Summary': 'Tom and Anna are excited to go on a holiday with their parents, and they fly on a big plane to a place with sun and sand.',
 'Story': 'Tom and Anna are brother and sister. They like to play with their toys and read books. They are very happy because they are going on a holiday with their mum and dad. They will fly on a big plane to a place with a lot of sun and sand.\nThe day of the holiday comes and they pack their bags. They go to the airport and wait for their plane. They see many other planes flying in the sky. They are very excited and want to fly too.\n"Look, Anna, that plane is so big and fast!" Tom says.\n"Yes, Tom, and it has wings and a tail. I wonder where it is going," Anna says.\nThey hear their mum call them. "Come on, kids, it\'s time to board our plane. We have to show our tickets and go through the gate."\nThey follow their mum and dad and get on their plane. They fi

In [153]:
# Story check so that completion exists
# and at least one other key in prompt, otherwise prompt is just Story.\n\n
records = [f for f in parsed if f.get("Story") and any(f[k] for k in f if k != "Story")]
len(records)          # 25026 — one record has no story

25026

In [154]:
def render(f):
    prompt = "".join(f"{k}: {f[k]}\n" for k in f if k != "Story") + "Story.\n\n"
    return prompt, f["Story"] + "\n<|endoftext|>\n"
prompt, completion = render(records[1])
print(prompt)

Features: Dialogue, BadEnding
Summary: Ben and Mia get lost in a subway station after a train zooms past them, leaving them alone and scared.
Random sentence: They wished they had never been eager to ride the subway.
Story.




In [155]:
print(completion)

Ben and Mia were eager to ride the subway with their mom. They had never been on a subway before. They wanted to see the big trains and the tunnels and the people.
Mom bought the tickets and they went to the platform. They saw a subway coming. It was loud and fast. Ben and Mia held mom's hands. They waited for the subway to stop.
But the subway did not stop. It zoomed past them. Ben and Mia were scared. They let go of mom's hands and ran away. They did not see where they were going. They got lost in the crowd.
Mom shouted, "Ben! Mia! Stop! Come back!" But they did not hear her. They did not see her. They were alone and afraid.
Mom looked for them. She asked the people. She asked the workers. She asked the police. But no one had seen them. She cried and cried. She did not know what to do.
Ben and Mia did not find mom. They did not find the subway. They did not find their way home. They had a bad day. They wished they had never been eager to ride the subway.
<|endoftext|>



In [156]:
from main import tok

In [157]:
ip, ic = tok.encode(prompt), tok.encode(completion)
ij = tok.encode(prompt + completion)
print(len(ip), len(ic), len(ip) + len(ic), len(ij))
print(tok.encode(prompt) + tok.encode(completion) == tok.encode(prompt + completion))

71 259 330 330
True


In [158]:
s = prompt[-20:] + completion[:20]
for id in tok.encode(s):
    print(f'{repr(tok.decode([id]))}')

't'
'he'
' su'
'b'
'way'
'.\n'
'S'
't'
'ory'
'.\n\n'
'Ben'
' and'
' Mia'
' were'
' eag'


In [159]:
def encode_example(f):
    p, c = render(f)
    ip = tok.encode(p)
    ic = tok.encode(c)
    return ip + ic, [False] * len(ip) + [True] * len(ic)

In [160]:
import torch
ids, is_c = encode_example(records[1])
x    = torch.tensor(ids[:-1])
y    = torch.tensor(ids[1:])
keep = torch.tensor(is_c[1:])
tok.decode(y[keep].tolist()) == render(records[1])[1]

True

In [161]:
def pack(records, block_size=512):
    ids, is_c = [], []
    for f in records:
        a, b = encode_example(f)
        ids += a; is_c += b
        if len(ids) >= block_size + 1:
            break
    ids, is_c = ids[:block_size + 1], is_c[:block_size + 1]
    x    = torch.tensor(ids[:-1])
    y    = torch.tensor(ids[1:]).masked_fill(~torch.tensor(is_c[1:]), -100)
    return x, y

x, y = pack(records[1:])
(y != -100).sum().item(), y.numel()

(381, 512)

In [162]:
seg = (y != -100).int()
print("".join(str(v) for v in seg.tolist()))


00000000000000000000000000000000000000000000000000000000000000000000001111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111000000000000000000000000000000000000000000000000000000000000011111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111


In [163]:
ids, is_c = encode_example(records[1])
[tok.decode([i]) for i in ids[-6:]]

['b', 'way', '.\n', '<|', 'endoftext', '|>\n']

In [164]:
from main import ROOT, device, GPT, GPTConfig
CKPT_DIR = ROOT / "artifacts" / "checkpoints"

In [165]:
ckpt = CKPT_DIR / "big_2026-08-16_06-45-06.pt"
saved = torch.load(ckpt, map_location=device)
reloaded = GPT(GPTConfig(**saved["config"])).to(device)
reloaded.load_state_dict(saved["model"])

<All keys matched successfully>

In [166]:
def generate_sample(prompt: str):
    idx = torch.tensor([tok.encode(prompt)], device=device)
    sample = tok.decode(
        reloaded.generate(
            idx,
            max_new_tokens=reloaded.cfg.block_size + 1 - idx.size(1),
            use_cache=False,
            temperature=0.8,
            top_p=0.95,
        )[0].tolist()
    )
    print(sample)

In [167]:
prompt, completion = render(records[0])
generate_sample(prompt)

Random sentence: They are very excited and want to fly too.
Features: Dialogue
Summary: Tom and Anna are excited to go on a holiday with their parents, and they fly on a big plane to a place with sun and sand.
Story.

The next day, Tom and Anna get on a plane with their parents and fly to the land of mountains. They see mountains, fields, forests and deserts. They see other people and animals.

Tom and Anna are amazed by the view. They thank their parents for taking them there. They thank their parents for taking them there. They say goodbye to the nice people.

Tom and Anna are very happy. They have had a wonderful holiday. They can't wait to go on another holiday. They hug their parents and their holiday. They are not anxious anymore.
<|endoftext|>
Lily and Ben are friends. They like to play in the park. One day, they find a big stick on the ground. Lily wants to touch it, but Ben says no.

"Please, Ben, let us touch the stick. It is not ours. It is for grown-ups," Lily says.

"No, L

In [168]:
prompt, completion = render(records[1])
ip = tok.encode(prompt)
ic = tok.encode(completion)
len(ip), len(ic)

(71, 259)

In [169]:
from einops import rearrange
import torch.nn.functional as F
ids, is_c = encode_example(records[1])
x    = torch.tensor([ids[:-1]], device=device)
y    = torch.tensor([ids[1:]],  device=device)
keep = torch.tensor([is_c[1:]], device=device)

with torch.no_grad():
    logits, _ = reloaded(x)
flat = rearrange(logits, "b t v -> (b t) v")
ce = lambda t: F.cross_entropy(
    flat, rearrange(t, "b t -> (b t)"), ignore_index=-100
).item()

ce(y), ce(y.masked_fill(~keep, -100)), ce(y.masked_fill(keep, -100))

(2.0411531925201416, 1.2923145294189453, 4.811856746673584)

(70 * 4.8119 + 259 * 1.2923) / 329 = 2.0412

notice the completion number: 1.2923, below the base checkpoint's 1.3784 val loss. The model has never been finetuned, yet conditioning on a summary makes stories easier to predict than unconditional TinyStories text. The instruction block is doing real work already.

In [170]:
with torch.no_grad():
    lg, _ = reloaded(torch.tensor([tok.encode(prompt)], device=device))
p, i = lg[0, -1].softmax(-1).topk(8)
[(tok.decode([j]), round(v, 3)) for v, j in zip(p.tolist(), i.tolist())]


[('The', 0.245),
 ('Ben', 0.223),
 ('Suddenly', 0.05),
 ('F', 0.046),
 ('But', 0.029),
 ('S', 0.028),
 ('A', 0.025),
 ('They', 0.024)]

In [171]:
opt = torch.optim.AdamW(reloaded.parameters(), lr=1e-5)
logits, _ = reloaded(x)
loss = F.cross_entropy(
    rearrange(logits, "b t v -> (b t) v"),
    rearrange(y.masked_fill(~keep, -100), "b t -> (b t)"),
    ignore_index=-100,
)
opt.zero_grad(set_to_none=True)
loss.backward()
opt.step()
print(loss.item())
logits, _ = reloaded(x)
loss = F.cross_entropy(
    rearrange(logits, "b t v -> (b t) v"),
    rearrange(y.masked_fill(~keep, -100), "b t -> (b t)"),
    ignore_index=-100,
)
print(loss.item())

1.2923145294189453
1.0796164274215698
